# Module 11 — Batch Predictions

Send **many non-urgent prompts in one job** for **higher throughput and lower cost**.

Batch is **different** from the prior modules: it is a **Vertex batch prediction job** with **GCS/BigQuery I/O**, submitted via the **Google Gen AI SDK** (`from google import genai`) — **not** the synchronous `AnthropicVertex` client. It still targets **Claude on the Agent Platform**, just through the batch interface. This builds on **Modules 00–10**.

## Part B — Bootstrap (batch-specific)

This module differs from the rest of the series in two ways:

- It uses a **REGIONAL** endpoint — the **`global`** endpoint is **not supported** for partner-model batch.
- It uses the **Google Gen AI SDK** configured for Vertex, not `AnthropicVertex`.

Auth is still **ADC** — no API key.

In [ ]:
%pip install --quiet google-genai google-cloud-storage

In [ ]:
# ===== BATCH CONFIG =====
import subprocess
from google import genai

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID     = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
# NOTE: 'global' is NOT supported for partner-model batch — use a REGIONAL location.
# Confirm a region that serves Opus 4.8 on the Model Garden card.
BATCH_LOCATION = "us-east5"
MODEL          = "claude-opus-4-8"
BUCKET         = "<YOUR_BUCKET>"      # GCS bucket name (no gs:// prefix)

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)
assert BUCKET and BUCKET != "<YOUR_BUCKET>", (
    "BUCKET is still the placeholder. Set it to an existing GCS bucket you own "
    "(name only, no gs:// prefix)."
)

# Google Gen AI SDK configured for Vertex (ADC auth).
gclient = genai.Client(vertexai=True, project=PROJECT_ID, location=BATCH_LOCATION)
print(f"✅ Gen AI (Vertex) client ready (project={PROJECT_ID}, location={BATCH_LOCATION}, model={MODEL}).")

## Part C — Build the input (GCS JSONL)

Each batch request is **one BigQuery row** or **one GCS JSONL line** in this exact shape:

```json
{"custom_id": "request-1",
 "request": {"messages": [{"role": "user", "content": "..."}],
             "anthropic_version": "vertex-2023-10-16",
             "max_tokens": 256}}
```

We'll write three requests to a local JSONL file and upload it to GCS.

In [ ]:
import json
from google.cloud import storage

prompts = [
    "Explain what a load balancer does in one sentence.",
    "Name three benefits of caching, as a comma-separated list.",
    "In one sentence, what is Application Default Credentials (ADC)?",
]

requests = [
    {
        "custom_id": f"request-{i+1}",
        "request": {
            "messages": [{"role": "user", "content": p}],
            "anthropic_version": "vertex-2023-10-16",
            "max_tokens": 256,
        },
    }
    for i, p in enumerate(prompts)
]

LOCAL_INPUT = "batch_input.jsonl"
with open(LOCAL_INPUT, "w", encoding="utf-8") as f:
    for r in requests:
        f.write(json.dumps(r) + "\n")  # one JSON object per line

# Upload to GCS. Assumes BUCKET already exists; to create it:
#   gsutil mb -l us-east5 gs://<YOUR_BUCKET>
INPUT_BLOB = "claude-batch/batch_input.jsonl"
INPUT_URI = f"gs://{BUCKET}/{INPUT_BLOB}"

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)
bucket.blob(INPUT_BLOB).upload_from_filename(LOCAL_INPUT)
print(f"✅ Uploaded input: {INPUT_URI}")

## Part D — Submit + poll the batch job

Submit with `gclient.batches.create`. Batch is **asynchronous** — completion can take **minutes to hours** (there's an SLA window) — so we poll the job state with a sleep and a max-wait timeout. The job keeps running server-side, so you can also check it later by name instead of blocking.

In [ ]:
import time

OUTPUT_URI = f"gs://{BUCKET}/claude-batch/output/"

# IMPORTANT: the exact create() config and the partner-model `model` reference
# should track the official doc "Batch predictions with Anthropic Claude models":
#   https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/partner-models/claude/batch
# Partner Claude models are referenced via the publisher path; confirm the exact
# form your SDK version expects in that doc.
MODEL_REF = f"publishers/anthropic/models/{MODEL}"

job = gclient.batches.create(
    model=MODEL_REF,
    src=INPUT_URI,
    config={"dest": OUTPUT_URI},
)
print(f"Submitted batch job: {job.name}")

TERMINAL = {
    "JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED",
    "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED",
}
MAX_WAIT_S = 600   # cap how long we block here; the job continues server-side.
POLL_S = 20

start = time.monotonic()
last_state = None
while True:
    job = gclient.batches.get(name=job.name)
    state = str(getattr(job.state, "name", job.state))
    if state != last_state:
        print(f"  state: {state}")
        last_state = state
    if state in TERMINAL:
        break
    if time.monotonic() - start > MAX_WAIT_S:
        print(f"  Still running after {MAX_WAIT_S}s — re-check later instead of blocking.")
        print(f"  Re-check with: gclient.batches.get(name='{job.name}')")
        break
    time.sleep(POLL_S)

## Part E — Read results

Results land in the **output location**, with a per-request **`response`** and **`status`**, keyed by **`custom_id`** (`response` and `status` are reserved field names). We read the output JSONL from GCS and print a snippet per request.

In [ ]:
state = str(getattr(job.state, "name", job.state))

if state == "JOB_STATE_SUCCEEDED":
    prefix = f"claude-batch/output/"
    blobs = [b for b in gcs.list_blobs(BUCKET, prefix=prefix) if b.name.endswith(".jsonl")]
    if not blobs:
        print("Job succeeded but no .jsonl output found yet under", OUTPUT_URI)
    for blob in blobs:
        for line in blob.download_as_text().splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            cid = row.get("custom_id", "(no id)")
            status = row.get("status", "(no status)")
            snippet = json.dumps(row.get("response", {}))[:200]
            print(f"\n[{cid}] status={status}")
            print(f"  response: {snippet} ...")
else:
    print(f"Job not complete (state={state}).")
    print(f"Re-check later with: gclient.batches.get(name='{job.name}')")
    print(f"When succeeded, read JSONL under: {OUTPUT_URI}")

## Part F — Notes & recap

### Notes

- **Regional only:** the **`global`** endpoint is **not supported** for partner-model batch — use a regional location that serves Opus 4.8 (confirm on the Model Garden card).
- **Quota:** default of **4 concurrent batch jobs** per project.
- **I/O:** input/output via **BigQuery** or **GCS JSONL**; output reserves the **`response`** and **`status`** field names, keyed by `custom_id`.
- **When to use:** non-urgent bulk work where **cost/throughput** matter more than latency.
- Confirm regions, quotas, and the exact SDK signature in the doc: https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/partner-models/claude/batch

### Recap

- Batch is a **Vertex batch prediction job** via the **Google Gen AI SDK** (GCS/BigQuery I/O), still targeting Claude on the Agent Platform — not the sync client.
- Input rows/lines use `{custom_id, request{messages, anthropic_version: "vertex-2023-10-16", max_tokens}}`.
- Submit with `gclient.batches.create(...)`, **poll** the state (async; minutes to hours), then read per-request **`response`**/**`status`** from the output keyed by `custom_id`.
- **Regional only**, default **4 concurrent jobs** — confirm details in the official doc.

**Next: Module 12 — Usage types (Provisioned Throughput & Shared Model Lineage Quota).**